# Minimum Edit Distance Deep Dive (SNLP Note 03)

Companion notebook for `03-Minimum-Edit-Distance.md` (Lecture 3, *Lexical Processing Part 2*). Every number quoted in the note comes from a cell here. Pure Python + numpy + matplotlib, nothing to download.

| Part | Topic |
|---|---|
| A | The DP table, built row by row (the slide's *leda → deal*) |
| B | *intention → execution*: distance 5 (sub = 1) and 8 (sub = 2) |
| C | Backtrace: read off the alignment, count optimal alignments |
| D | Why brute force is hopeless: counting edit paths |
| E | Spell correction: *graffe* and its candidates |
| F | Word-level edit distance: WER for the MT/ASR slide |
| G | DNA alignment (the biology slide) |
| H | Weighted edit distance with a keyboard-distance cost |
| I | Space-saving version (two rows) and timing |
| J | Checks for the practice problems |

In [1]:
import time
import numpy as np

def med_table(x, y, sub_cost=1, ins_cost=1, del_cost=1):
    '''D[i][j] = edit distance between x[:i] and y[:j]. Returns the full (n+1) x (m+1) table.'''
    n, m = len(x), len(y)
    D = np.zeros((n + 1, m + 1), dtype=float)
    for i in range(1, n + 1):
        D[i, 0] = D[i - 1, 0] + del_cost          # delete every letter of x[:i]
    for j in range(1, m + 1):
        D[0, j] = D[0, j - 1] + ins_cost          # insert every letter of y[:j]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            s = 0 if x[i - 1] == y[j - 1] else sub_cost
            D[i, j] = min(D[i - 1, j] + del_cost,      # delete x_i
                          D[i, j - 1] + ins_cost,      # insert y_j
                          D[i - 1, j - 1] + s)         # copy / substitute
    return D

def med(x, y, sub_cost=1):
    return med_table(x, y, sub_cost)[-1, -1]

def show(D, x, y, flip=False):
    '''Print the table. flip=True prints it like the slides: row 0 (#) at the bottom.'''
    rows = list(range(len(x) + 1))
    if flip:
        rows = rows[::-1]
    for i in rows:
        lab = '#' if i == 0 else x[i - 1]
        print(f"{lab:>3} " + " ".join(f"{int(v):>3}" if v == int(v) else f"{v:>5.1f}" for v in D[i]))
    print("    " + " ".join(f"{c:>3}" for c in '#' + y))

## Part A — The table for *leda → deal* (slide example, all costs 1)

`D[i][j]` = cost of turning the first *i* letters of the source into the first *j* letters of the target. Row 0 and column 0 are the "start from nothing" costs; each other cell looks at three neighbours (up, left, diagonal).

In [2]:
x, y = "leda", "deal"
D = med_table(x, y)
for r in range(1, len(x) + 1):
    print(f"--- after row {r} ({x[r-1]!r}): {D[r].astype(int).tolist()}")
print()
show(D, x, y)
print("\nMED(leda, deal) =", int(D[-1, -1]))

--- after row 1 ('l'): [1, 1, 2, 3, 3]
--- after row 2 ('e'): [2, 2, 1, 2, 3]
--- after row 3 ('d'): [3, 2, 2, 2, 3]
--- after row 4 ('a'): [4, 3, 3, 2, 3]

  #   0   1   2   3   4
  l   1   1   2   3   3
  e   2   2   1   2   3
  d   3   2   2   2   3
  a   4   3   3   2   3
      #   d   e   a   l

MED(leda, deal) = 3


## Part B — *intention → execution*

The slide gives distance 5 when every operation costs 1 and 8 when a substitution costs 2 (a substitution = a deletion + an insertion). The second table is printed upside-down like the slide (`#` row at the bottom).

In [3]:
x, y = "intention", "execution"
print("sub = 1 :", int(med(x, y, 1)))
print("sub = 2 :", int(med(x, y, 2)))
print()
show(med_table(x.upper(), y.upper(), 2), x.upper(), y.upper(), flip=True)

sub = 1 : 5
sub = 2 : 8

  N   9   8   9  10  11  12  11  10   9   8
  O   8   7   8   9  10  11  10   9   8   9
  I   7   6   7   8   9  10   9   8   9  10
  T   6   5   6   7   8   9   8   9  10  11
  N   5   4   5   6   7   8   9  10  11  10
  E   4   3   4   5   6   7   8   9  10   9
  T   3   4   5   6   7   8   7   8   9   8
  N   2   3   4   5   6   7   8   7   8   7
  I   1   2   3   4   5   6   7   6   7   8
  #   0   1   2   3   4   5   6   7   8   9
      #   E   X   E   C   U   T   I   O   N


## Part C — Backtrace and alignment

Keep a pointer for every cell (which neighbour gave the minimum). From the top-right corner `D(n,m)` follow the pointers back to `D(0,0)`; each step is one column of the alignment.

In [4]:
def backtrace(x, y, sub_cost=1):
    D = med_table(x, y, sub_cost)
    i, j, ops = len(x), len(y), []
    while i > 0 or j > 0:
        s = 0 if i and j and x[i-1] == y[j-1] else sub_cost
        if i and j and D[i, j] == D[i-1, j-1] + s:      # prefer diagonal
            ops.append(('copy' if s == 0 else 'sub', x[i-1], y[j-1])); i, j = i-1, j-1
        elif i and D[i, j] == D[i-1, j] + 1:
            ops.append(('del', x[i-1], '*')); i -= 1
        else:
            ops.append(('ins', '*', y[j-1])); j -= 1
    return ops[::-1]

def show_alignment(x, y, sub_cost=1):
    ops = backtrace(x, y, sub_cost)
    code = {'copy': ' ', 'sub': 's', 'del': 'd', 'ins': 'i'}
    print(" ".join(a for _, a, _ in ops))
    print(" ".join('|' for _ in ops))
    print(" ".join(b for _, _, b in ops))
    print(" ".join(code[o] for o, _, _ in ops))
    cost = sum({'copy': 0, 'sub': sub_cost, 'del': 1, 'ins': 1}[o] for o, _, _ in ops)
    print("cost =", cost)

show_alignment("intention", "execution", 2)
print()
show_alignment("intention", "execution", 1)

i n t e * n t i o n
| | | | | | | | | |
* e x e c u t i o n
d s s   i s        
cost = 8

i n t e n t i o n
| | | | | | | | |
e x e c u t i o n
s s s s s        
cost = 5


In [5]:
def count_optimal(x, y, sub_cost=1):
    '''Number of distinct optimal edit paths through the table.'''
    D = med_table(x, y, sub_cost); n, m = len(x), len(y)
    C = np.zeros((n + 1, m + 1), dtype=object); C[0, 0] = 1
    for i in range(n + 1):
        for j in range(m + 1):
            if i == j == 0: continue
            c = 0
            if i and D[i, j] == D[i-1, j] + 1: c += C[i-1, j]
            if j and D[i, j] == D[i, j-1] + 1: c += C[i, j-1]
            if i and j:
                s = 0 if x[i-1] == y[j-1] else sub_cost
                if D[i, j] == D[i-1, j-1] + s: c += C[i-1, j-1]
            C[i, j] = c
    return C[n, m]

for pair in [("intention", "execution"), ("leda", "deal"), ("kitten", "sitting")]:
    print(pair, "optimal paths: sub=1 ->", count_optimal(*pair, 1), " sub=2 ->", count_optimal(*pair, 2))

('intention', 'execution') optimal paths: sub=1 -> 7  sub=2 -> 134
('leda', 'deal') optimal paths: sub=1 -> 2  sub=2 -> 4
('kitten', 'sitting') optimal paths: sub=1 -> 1  sub=2 -> 9


## Part D — Why not just search all edit sequences?

A path through the table moves right (insert), up (delete) or diagonally (substitute/copy). The number of such paths from `(0,0)` to `(n,m)` is the **Delannoy number**. The DP table has only `(n+1)(m+1)` cells.

In [6]:
def delannoy(n, m):
    T = [[1] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            T[i][j] = T[i-1][j] + T[i][j-1] + T[i-1][j-1]
    return T[n][m]

for n in [4, 9, 15, 20, 30]:
    print(f"n = m = {n:2d}: paths = {delannoy(n, n):>22,}   table cells = {(n+1)**2:>4}")

n = m =  4: paths =                    321   table cells =   25
n = m =  9: paths =              1,462,563   table cells =  100
n = m = 15: paths =         44,642,381,823   table cells =  256
n = m = 20: paths =    260,543,813,797,441   table cells =  441
n = m = 30: paths = 9,642,641,465,118,083,682,429   table cells =  961


## Part E — Spell correction: the user typed *graffe*

In [7]:
typo = "graffe"
cands = ["graf", "graft", "grail", "giraffe"]
for c in cands:
    print(f"{c:8s} sub=1: {int(med(typo, c, 1))}   sub=2: {int(med(typo, c, 2))}")
print()
show_alignment("graffe", "giraffe", 1)

graf     sub=1: 2   sub=2: 2
graft    sub=1: 2   sub=2: 3
grail    sub=1: 3   sub=2: 5
giraffe  sub=1: 1   sub=2: 1

g * r a f f e
| | | | | | |
g i r a f f e
  i          
cost = 1


## Part F — Word-level edit distance (MT / speech recognition)

Same algorithm, but the "letters" are words. Word Error Rate = (S + D + I) / number of reference words.

In [8]:
R = "Spokesman confirms senior government adviser was shot".split()
H = "Spokesman said the senior adviser was shot dead".split()
ops = backtrace(R, H, 1)
for o, a, b in ops:
    print(f"{o:5s} {a:12s} {b}")
S = sum(o == 'sub' for o, _, _ in ops); Dl = sum(o == 'del' for o, _, _ in ops); I = sum(o == 'ins' for o, _, _ in ops)
print(f"\nS={S} D={Dl} I={I}  edits={S+Dl+I}  reference words={len(R)}  WER={(S+Dl+I)/len(R):.1%}")
# The slide's alignment (S, I, D, I) costs the same 4 edits: a tie, so both are optimal.
print("optimal alignments of R and H:", count_optimal(R, H, 1))

copy  Spokesman    Spokesman
sub   confirms     said
sub   senior       the
sub   government   senior
copy  adviser      adviser
copy  was          was
copy  shot         shot
ins   *            dead

S=3 D=0 I=1  edits=4  reference words=7  WER=57.1%
optimal alignments of R and H: 3


## Part G — The biology slide

The two DNA strings from the slide. Biologists use *similarity scores* (Needleman–Wunsch) rather than a cost, but the table is the same idea.

In [9]:
a = "AGGCTATCACCTGACCTCCAGGCCGATGCCC"
b = "TAGCTATCACGACCGCGGTCGATTTGCCCGAC"
print(len(a), len(b))
print("edit distance (unit costs):", int(med(a, b, 1)))
ops = backtrace(a, b, 1)
print("".join(p for _, p, _ in ops).replace('*', '-'))
print("".join(q for _, _, q in ops).replace('*', '-'))
# The slide's alignment, scored with unit costs
top = "-AGGCTATCACCTGACCTCCAGGCCGA--TGCCC---"
bot = "TAG-CTATCAC--GACCGC--GGTCGATTTGCCCGAC"
cost = sum(1 for p, q in zip(top, bot) if p != q)
print("slide's alignment, unit-cost score:", cost, "(gaps:", top.count('-') + bot.count('-'), ")")

31 32
edit distance (unit costs): 13
AGGCTATCACCTGACCTCCAGGCCGA--TG-CC--C
TAGCTATCA-C-GACC-GC-GGTCGATTTGCCCGAC
slide's alignment, unit-cost score: 13 (gaps: 11 )


## Part H — Weighted edit distance: cheaper substitutions for neighbouring keys

The slide's idea: some mistakes are more likely (e.g. hitting a neighbouring key). Give those substitutions a smaller cost. Here: substitution between keys that touch on a QWERTY keyboard = 0.5, any other substitution = 1, insert/delete = 1.

In [10]:
ROWS = ["qwertyuiop", "asdfghjkl", "zxcvbnm"]
pos = {c: (r, k + 0.5 * r) for r, row in enumerate(ROWS) for k, c in enumerate(row)}
def adjacent(a, b):
    (r1, c1), (r2, c2) = pos[a], pos[b]
    return abs(r1 - r2) <= 1 and abs(c1 - c2) <= 1.0 and a != b

def wmed(x, y, near=0.5):
    n, m = len(x), len(y)
    D = np.zeros((n + 1, m + 1))
    D[:, 0] = np.arange(n + 1); D[0, :] = np.arange(m + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            a, b = x[i-1], y[j-1]
            s = 0 if a == b else (near if adjacent(a, b) else 1)
            D[i, j] = min(D[i-1, j] + 1, D[i, j-1] + 1, D[i-1, j-1] + s)
    return D[n, m]

print("neighbours of 'e':", sorted(c for c in pos if adjacent('e', c)))
for typo, cands in [("thw", ["the", "thy", "two"]), ("hwllo", ["hello", "hallo", "hullo"])]:
    print(f"\n{typo}:")
    for c in cands:
        print(f"  {c:8s} plain: {med(typo, c, 1):.1f}   keyboard-weighted: {wmed(typo, c):.1f}")

neighbours of 'e': ['d', 'r', 's', 'w']

thw:
  the      plain: 1.0   keyboard-weighted: 0.5
  thy      plain: 1.0   keyboard-weighted: 1.0
  two      plain: 2.0   keyboard-weighted: 2.0

hwllo:
  hello    plain: 1.0   keyboard-weighted: 0.5
  hallo    plain: 1.0   keyboard-weighted: 0.5
  hullo    plain: 1.0   keyboard-weighted: 1.0


## Part I — Two rows are enough (if you only need the distance)

Row *i* only needs row *i − 1*, so the distance can be computed in O(m) memory. The alignment needs the whole table (or Hirschberg's divide-and-conquer trick).

In [11]:
def med_two_rows(x, y):
    prev = list(range(len(y) + 1))
    for i, a in enumerate(x, 1):
        cur = [i]
        for j, b in enumerate(y, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (a != b)))
        prev = cur
    return prev[-1]

rng = np.random.default_rng(0)
for n in [200, 400, 800]:
    s = "".join(rng.choice(list("acgt"), n)); t = "".join(rng.choice(list("acgt"), n))
    t0 = time.perf_counter(); d = med_two_rows(s, t); dt = time.perf_counter() - t0
    print(f"n = m = {n}: distance {d}, {dt*1000:.0f} ms  (cells: {n*n:,})")
print("doubling n -> about 4x the time: O(nm)")

n = m = 200: distance 108, 6 ms  (cells: 40,000)
n = m = 400: distance 217, 21 ms  (cells: 160,000)
n = m = 800: distance 418, 99 ms  (cells: 640,000)
doubling n -> about 4x the time: O(nm)


## Part J — Checks for the practice problems

In [12]:
checks = [("leda", "deal"), ("sunday", "saturday"), ("drive", "brief"), ("drive", "divers"), ("kitten", "sitting"),
          ("rain", "shine"), ("execution", "intention"), ("abc", ""), ("", "abc"), ("algorithm", "altruistic"), ("form", "from")]
for x, y in checks:
    print(f"{x!r:12} -> {y!r:12} sub=1: {int(med(x, y, 1))}   sub=2: {int(med(x, y, 2))}")
print()
show(med_table("drive", "brief"), "drive", "brief")
print()
show_alignment("drive", "divers")
print()
show(med_table("rain", "shine"), "rain", "shine")

'leda'       -> 'deal'       sub=1: 3   sub=2: 4
'sunday'     -> 'saturday'   sub=1: 3   sub=2: 4
'drive'      -> 'brief'      sub=1: 3   sub=2: 4
'drive'      -> 'divers'     sub=1: 3   sub=2: 3
'kitten'     -> 'sitting'    sub=1: 3   sub=2: 5
'rain'       -> 'shine'      sub=1: 3   sub=2: 5
'execution'  -> 'intention'  sub=1: 5   sub=2: 8
'abc'        -> ''           sub=1: 3   sub=2: 3
''           -> 'abc'        sub=1: 3   sub=2: 3
'algorithm'  -> 'altruistic' sub=1: 6   sub=2: 9
'form'       -> 'from'       sub=1: 2   sub=2: 2

  #   0   1   2   3   4   5
  d   1   1   2   3   4   5
  r   2   2   1   2   3   4
  i   3   3   2   1   2   3
  v   4   4   3   2   2   3
  e   5   5   4   3   2   3
      #   b   r   i   e   f

d r i v e * *
| | | | | | |
d * i v e r s
  d       i i
cost = 3

  #   0   1   2   3   4   5
  r   1   1   2   3   4   5
  a   2   2   2   3   4   5
  i   3   3   3   2   3   4
  n   4   4   4   3   2   3
      #   s   h   i   n   e
